# 10 · Batch Processing — The Derived Data Factory

**Part 1 · Data-Intensive Applications** · *Prerequisites: 03, 06* · *Book mapping: DDIA ch. 10*

So far, services: requests in, responses out, latency the metric. **Batch processing** is the
other computational lifestyle: take a *bounded* dataset, chew through all of it, produce a
*derived* dataset — search indexes, recommendations, reports, ML training data (all of
Part 2 eats batch output). Throughput is the metric; jobs are restartable; input is
immutable and output is rebuildable — a property with deep consequences (nb 12 is built on
it).

**Learning objectives.**

1. See the Unix philosophy as the original batch architecture and know why it scales.
2. Build a working **MapReduce** engine — map, partition, shuffle, sort, reduce — with
   fault-tolerant task retry, and run real jobs on it.
3. Implement the two classic distributed **joins** (reduce-side and map-side) and know when
   each applies.
4. Build a mini **dataflow engine** and measure exactly what Spark-style pipelining saves
   over chained MapReduce jobs.

In [ ]:
import hashlib
import heapq
import json
import subprocess
from collections import Counter
from itertools import groupby
from operator import itemgetter

import numpy as np
import pandas as pd

from sysdes import scratch_dir, viz

viz.use_style()
rng = np.random.default_rng(seed=10)
WORK = scratch_dir("10-batch-processing")

def stable_hash(s: str) -> int:
    return int.from_bytes(hashlib.blake2b(str(s).encode(), digest_size=8).digest())

## 1. The Unix ancestors

Batch processing was invented twice, and the first version still runs your laptop: small
programs, each doing one thing, composed through a **uniform interface** (streams of text
lines) with **pipes**. Analyze a web server log — top 5 URLs — in one line of shell:

In [ ]:
# Synthesize an access log with deliberately distinct URL frequencies:
URLS = ["/home", "/about", "/post/1", "/post/2", "/api/health", "/login"]
WEIGHTS = np.array([0.34, 0.05, 0.24, 0.14, 0.20, 0.03])
lines = [f'203.0.113.9 - - [30/Aug/2026] "GET {u} HTTP/1.1" 200 512'
         for u in rng.choice(URLS, size=20_000, p=WEIGHTS)]
log_path = WORK / "access.log"
log_path.write_text("\n".join(lines) + "\n")

pipeline = f"cut -d' ' -f6 {log_path} | sort | uniq -c | sort -rn | head -5"
out = subprocess.run(["bash", "-c", pipeline], capture_output=True, text=True,
                     env={"LC_ALL": "C", "PATH": "/usr/bin:/bin"}, check=True)
print(f"$ {pipeline}\n{out.stdout}")

unix_top5 = {(fields := line.split())[1]: int(fields[0])
             for line in out.stdout.strip().splitlines()}
py_top5 = dict(Counter(line.split()[5] for line in lines).most_common(5))
assert unix_top5 == py_top5, "the shell pipeline and Python agree"

Why this composes so well — the properties every batch system since has copied:

- **Immutable inputs**: no tool damages the log; you can rerun endlessly (experiments are
  free, and *rerunning is the whole fault-tolerance strategy* later).
- **Uniform interface**: any output can be any input; tools don't know their neighbors.
- **`sort` does external merge sort**: it handles data far larger than RAM by spilling
  sorted runs and merging — literally the SSTable merge from nb 03. Sorting is the
  workhorse that makes same-key records adjacent, which is what `uniq -c` (and every
  reducer below) depends on.

What Unix pipes lack: multiple machines. **MapReduce is pipes across a cluster** — with the
sort built in.

## 2. Build: a mini-MapReduce

The programming model: you supply two callbacks —

- `map_fn(record) → yields (key, value)` — extract/transform, run in parallel over input
  **splits**;
- `reduce_fn(key, values) → yields results` — aggregate, run in parallel over keys.

The engine owns everything between: **partition** each mapper's output by `hash(key)` into
one bucket per reducer (nb 06's hash partitioning, verbatim), write each bucket **sorted**
(one sequential file, nb 03), then each reducer **merges** its buckets from all mappers
(heapq again) and walks groups of equal keys. The mapper→reducer handoff is the famous
**shuffle**. Fault tolerance = tasks are deterministic and side-effect-free, so the engine
simply **re-runs failed tasks**:

In [ ]:
class TaskFailure(Exception):
    """A simulated worker crash mid-task."""

def run_mapreduce(splits, map_fn, reduce_fn, n_reducers=4, workdir=WORK,
                  fail_once: set | None = None, stats: dict | None = None):
    """Execute a MapReduce job. `fail_once`: map-task ids that crash on their
    first attempt (the engine must retry them). `stats` collects counters."""
    stats = stats if stats is not None else {}
    stats.setdefault("map_records_written", 0)
    stats.setdefault("task_retries", 0)
    failed_already: set[int] = set()

    # ---- MAP PHASE: each split -> n_reducers sorted partition files
    for m, split in enumerate(splits):
        for attempt in range(2):                      # at-most-one retry, for the demo
            try:
                buckets = [[] for _ in range(n_reducers)]
                for record in split:
                    for key, value in map_fn(record):
                        buckets[stable_hash(key) % n_reducers].append((key, value))
                if fail_once and m in fail_once and m not in failed_already:
                    failed_already.add(m)
                    raise TaskFailure(f"map task {m} died (simulated)")
                for r, bucket in enumerate(buckets):  # write SORTED partition files
                    bucket.sort(key=itemgetter(0))
                    with open(workdir / f"map{m}-part{r}.jsonl", "w") as f:
                        for kv in bucket:
                            f.write(json.dumps(kv) + "\n")
                    stats["map_records_written"] += len(bucket)
                break
            except TaskFailure as e:
                # Determinism + immutable input = retry is always safe.
                stats["task_retries"] += 1
                print(f"  engine: {e} -> retrying")

    # ---- REDUCE PHASE: merge this reducer's bucket from EVERY mapper (shuffle),
    # then walk groups of equal keys - possible only because buckets are sorted.
    output = []
    for r in range(n_reducers):
        streams = []
        for m in range(len(splits)):
            path = workdir / f"map{m}-part{r}.jsonl"
            streams.append(tuple(json.loads(line) for line in open(path)))
        merged = heapq.merge(*streams, key=itemgetter(0))
        for key, group in groupby(merged, key=itemgetter(0)):
            output.extend(reduce_fn(key, [v for _, v in group]))
    return output, stats

# ---- Job 1: word count, with a worker crash injected.
docs = [f"doc {i}: " + " ".join(rng.choice(["data", "systems", "scale", "logs",
                                            "batch", "shuffle"], size=12))
        for i in range(2_000)]
splits = [docs[i::4] for i in range(4)]              # 4 input splits = 4 map tasks

def wc_map(doc):
    for word in doc.split():
        yield word, 1

def wc_reduce(word, counts):
    yield word, sum(counts)

result, stats = run_mapreduce(splits, wc_map, wc_reduce, fail_once={2})
reference = Counter(w for d in docs for w in d.split())
assert dict(result) == dict(reference), "MapReduce agrees with single-machine Counter"
assert stats["task_retries"] == 1, "the crashed task was rerun, invisibly to the result"
print(f"word count over {len(docs)} docs OK despite a worker crash; "
      f"top: {Counter(dict(result)).most_common(3)}")

Notice what you did *not* write: parallelism, distribution, retry, sorting, or grouping.
That's the deal MapReduce offered in 2004: **write two pure functions; the engine turns a
machine-scale technique into a cluster-scale one** — the same hash partitioning (nb 06),
sorted runs and merges (nb 03), and re-execution-on-failure that Unix pioneered.

## 3. Joins, the batch way

Real jobs correlate datasets: events with user profiles, clicks with impressions. Per-event
lookups against a database would be nb 01's N+1 disaster at billion scale — so batch joins
restructure the *whole computation* instead.

### 3.1 Reduce-side (sort-merge) join

Map both datasets to the join key; the shuffle then delivers *everything about each user to
one reducer call* — profile and events together. Tag each record with its source so the
reducer can tell them apart:

In [ ]:
users = [{"user_id": i, "name": f"user{i:03d}", "city": ["oslo", "lima", "pune"][i % 3]}
         for i in range(300)]
events = [{"user_id": int(rng.integers(0, 300)), "action": a}
          for a in rng.choice(["view", "click", "buy"], size=5_000)]

def join_map(record):
    tag, row = record
    if tag == "users":
        yield row["user_id"], ("A_user", row["name"])   # 'A' sorts before 'B': the
    else:                                               # profile arrives first in
        yield row["user_id"], ("B_event", row["action"])  # each reducer group

def join_reduce(user_id, values):
    values.sort()                    # tiny per-user list; real MR uses secondary sort
    if not values or values[0][0] != "A_user":
        return                       # event for an unknown user: inner join drops it
    name = values[0][1]
    for tag, action in values[1:]:
        yield (user_id, name, action)

tagged = [("users", u) for u in users] + [("events", e) for e in events]
joined, _ = run_mapreduce([tagged[i::3] for i in range(3)], join_map, join_reduce)

expected = pd.DataFrame(events).merge(pd.DataFrame(users), on="user_id")
assert sorted(joined) == sorted(
    expected[["user_id", "name", "action"]].itertuples(index=False, name=None)), \
    "reduce-side join == pandas merge"
print(f"joined {len(joined)} event rows to profiles via shuffle")

### 3.2 Map-side (broadcast hash) join

If one side is *small* (fits in every mapper's RAM), skip the shuffle entirely: load the
small table into a dict, join inline while streaming the big one:

In [ ]:
users_by_id = {u["user_id"]: u["name"] for u in users}       # broadcast to every mapper

def broadcast_join_map(event):
    name = users_by_id.get(event["user_id"])
    if name is not None:
        yield (event["user_id"], name, event["action"]), None

def identity_reduce(key, values):
    # Keys came back through the JSON shuffle files as lists; restore tuples.
    for _ in values:
        yield tuple(key)

joined2, stats2 = run_mapreduce([events[i::3] for i in range(3)],
                                broadcast_join_map, identity_reduce)
assert sorted(joined2) == sorted(joined), "same join, no shuffle of the big side needed"
print("map-side join matches - the small table rode along in RAM")

Choosing: **reduce-side** always works (both sides arbitrarily large) and costs a full
shuffle+sort of everything; **map-side** costs nothing extra but requires a small side (or
pre-partitioned inputs). And the batch-join failure mode is an old friend: a **hot key**
(one user with 10M events — the linchpin celebrity) sends its whole group to one reducer,
and the job waits on that straggler. The cure is nb 06's, verbatim: salt the hot key,
replicate the small side across its salts. The skew never left; only the timescale changed.

## 4. From MapReduce to dataflow engines

MapReduce's tax: between chained jobs, the *entire intermediate dataset* is **materialized**
to disk (in the original, to a replicated filesystem!). Real workflows are 5–50 chained
stages — that's a lot of writing data just to read it right back. **Dataflow engines**
(Spark, Flink) treat the whole workflow as one graph and **pipeline** operators: a record
flows through parse→filter→map in memory, touching disk only at true shuffle boundaries.

Build the smallest possible one — lazy datasets as chained generators, with materialization
counted — and run the same 2-stage job both ways:

In [ ]:
class Dataset:
    """A lazily-evaluated dataset: transformations build a generator chain
    (pipelined, zero materialization); only shuffles write 'to disk'."""

    def __init__(self, source_fn, stats):
        self._source = source_fn        # callable -> fresh iterator
        self.stats = stats

    def map(self, f):
        return Dataset(lambda: (f(x) for x in self._source()), self.stats)

    def filter(self, pred):
        return Dataset(lambda: (x for x in self._source() if pred(x)), self.stats)

    def group_reduce(self, key_fn, reduce_fn):
        def shuffled():
            partitions: dict = {}
            for x in self._source():                 # the ONE pass over upstream
                partitions.setdefault(key_fn(x), []).append(x)
                self.stats["records_materialized"] += 1   # shuffle write
            for key, group in sorted(partitions.items()):
                yield key, reduce_fn(group)
        return Dataset(shuffled, self.stats)

    def collect(self):
        return list(self._source())

# The job: parse log lines -> keep only errors -> count per URL.
error_rate = 0.12
raw = [f"{u} {500 if rng.random() < error_rate else 200}"
       for u in rng.choice(URLS, size=30_000, p=WEIGHTS)]

# --- Dataflow: one pipelined pass; only the shuffle materializes (errors only).
df_stats = {"records_materialized": 0}
errors_by_url = (Dataset(lambda: iter(raw), df_stats)
                 .map(lambda line: line.split())
                 .filter(lambda rec: rec[1] == "500")
                 .group_reduce(lambda rec: rec[0], len)
                 .collect())

# --- Chained MapReduce: job 1 (parse+filter) writes ALL its output to disk,
#     job 2 (count) shuffles it to disk AGAIN.
mr_stats = {}
filtered, mr_stats = run_mapreduce(
    [raw[i::4] for i in range(4)],
    lambda line: iter([(line.split()[0], 1)]) if line.split()[1] == "500" else iter([]),
    lambda url, ones: [(url, len(ones))],
    stats=mr_stats)
# (We fused it into one job to be GENEROUS to MapReduce - a real chain would
# write the filtered dataset once more between the jobs.)

assert dict(errors_by_url) == dict(filtered), "identical answers"
n_errors = sum(dict(filtered).values())
print(f"errors: {n_errors:,} of {len(raw):,} lines")
print(f"records written to disk - MapReduce shuffle: {mr_stats['map_records_written']:,}, "
      f"dataflow shuffle: {df_stats['records_materialized']:,}")
assert df_stats["records_materialized"] == n_errors
assert mr_stats["map_records_written"] == n_errors, \
    "even fused, MR wrote the same shuffle - the win compounds with every extra stage"

The measured story: both engines must materialize the true shuffle (grouping needs all
matching records together — no way around it). The dataflow win is everything *else*:
parse→filter→map stages that MapReduce would write out between jobs flow through memory,
and with 10-stage pipelines that's 10× less disk traffic (plus: operators like sort can be
*chosen* per-stage rather than mandatory). The price: fault tolerance gets subtler — with
no materialized checkpoints, a lost partition must be **recomputed from lineage** (the
recorded chain of transformations), which is only possible because transformations are
deterministic and inputs immutable. The Unix property was load-bearing all along.

## 5. What batch is *for*

The output of a batch job is a **derived dataset**, and the healthiest pattern treats it
like a build artifact: outputs are written to a *new* location, verified, then atomically
swapped in — never edited in place. Canonical products:

- **Search indexes**: crawl → invert (word → posting list) → ship immutable index files
  (SSTables again); nb 12 wires this into a live system.
- **ML training data & features**: Part 2 (nb 14–15) consumes exactly this factory.
- **Recommendations / analytics / reports**: too expensive per-request; precompute.
- **ETL**: operational data reshaped for the warehouse (nb 03's OLAP columns).

If the job is wrong? Fix the code, rerun, replace the output. **Human fault tolerance
through rebuildability** — the property request-serving systems can only dream of, and the
reason nb 12 will argue for keeping raw immutable inputs forever.

## 6. Exercises

1. **Sessionization.** Design the MapReduce job that turns (user_id, timestamp, page) click
   records into *sessions* (gaps > 30 min split sessions). What is the map key? Where does
   the 30-minute logic run, and what ordering does the reducer need — and how does the
   *secondary sort* trick (putting the timestamp into the sort key) remove the need to
   buffer a user's whole history in reducer memory?
2. **Top-K.** Compute the 10 most-frequent words over 10 TB of text with two MapReduce
   jobs. Why can't one job do it? What does a **combiner** (a mini-reduce run inside each
   map task) save, and for which reduce functions is one legal?
3. **Skewed join.** User `celebrity` has 50M events; the users table is 10 GB (too big to
   broadcast whole). Combine nb 06's salting with a partial broadcast to unblock the
   reduce-side join. What exactly gets replicated, and by how much?
4. **Why sorted buckets?** Our mappers sort each partition file and reducers use
   `heapq.merge` + `groupby`. Suppose they instead shipped unsorted buckets and reducers
   used a hash map of key → values. What breaks first as data grows, and which nb 03
   idea is the sorted design reusing?
5. **In praise of materialization.** Give two concrete situations where MapReduce-style
   full materialization between stages beats pipelining, and the dataflow-engine features
   that answer them (checkpointing / persist).

### Solutions

**1.** Map: emit key `(user_id)` value `(timestamp, page)` — with secondary sort, the
*shuffle key* is `(user_id, timestamp)` while grouping remains by `user_id`, so each
reducer receives one user's clicks **already time-ordered** as a stream. The reducer walks
the stream holding only (previous timestamp, current session accumulator): gap > 30 min →
emit session, start new. Without secondary sort it must collect all a user's clicks and
sort in memory — fine for most users, fatal for the crawler bot with 80M clicks (skew,
again).

**2.** Job 1: word count (with a **combiner** = the reducer itself run map-side, legal
because `sum` is associative + commutative — it collapses each mapper's local duplicates,
shrinking shuffle traffic enormously for skewed word distributions). Job 2: single reducer
(or tree) over job 1's output computing top 10 — a *global* order needs one place where
all candidates meet; do it on the tiny counted dataset, not the raw one. Non-legal
combiner example: `mean` (not associative over partial groups — ship (sum, count) pairs
instead).

**3.** Split the join: (a) all non-hot keys → normal reduce-side join; (b) for the hot key:
salt the *events* to `celebrity#0..99` (spreading 50M events over 100 reducers) and
**replicate only the celebrity's single user row** to all 100 salts (tiny — you broadcast
one row, not the 10 GB table). Cost: 100× one profile row plus a routing rule; benefit: the
straggler reducer disappears.

**4.** The hash-map reducer holds **every value of every key in its partition in RAM at
once** — it breaks at the first partition whose group data exceeds memory (and skew makes
that one hot partition, early). The sorted design streams: `merge` needs one record per
input file in memory; `groupby` needs one group at a time. It is exactly nb 03's
SSTable-merge trick: sorting converts a memory problem into a sequential-IO problem.

**5.** (a) A 40-stage nightly pipeline where stage 23 flakes: with materialized stage
outputs you restart at 23, not at 0 — coarse-grained recovery beats lineage recomputation
when stages are expensive; dataflow answer: explicit **checkpointing**. (b) An intermediate
dataset consumed by *five* downstream jobs: materialize once, read five times, instead of
recomputing the lineage five times; dataflow answer: `persist()`/cache. Materialization
was never wrong — it was *mandatory*; engines made it a choice.

## 7. Takeaways

- Batch = bounded input → derived output; immutable inputs + deterministic tasks make
  **rerunning** the answer to faults (machine *and* human).
- MapReduce = Unix pipes at cluster scale: hash-partition (nb 06) + sorted runs and merges
  (nb 03) + pure functions. The shuffle is where the cluster's money goes.
- Joins are restructured, not looked up: sort-merge (universal) vs broadcast (small side);
  skew is the eternal enemy and salting the eternal cure.
- Dataflow engines make materialization optional (pipelining + lineage), turning 10 jobs
  into 1 graph; keep checkpoints where recovery or reuse demands them.
- Treat derived datasets as build artifacts: rebuild and swap, never patch in place.

**Further reading.** DDIA ch. 10; Dean & Ghemawat, *MapReduce* (OSDI 2004); Zaharia et
al., *Resilient Distributed Datasets* (NSDI 2012 — lineage, the Spark idea).

**Next:** [11 · Stream processing](11-stream-processing.ipynb) — batch's unbounded sibling:
what happens when the input never ends.